# IMDB Dataset playground

## Setup

In [1]:
import tensorflow as tf
import numpy as np


In [13]:
np.random.seed(42)
tf.random.set_seed(42)

## Data load

In [2]:
from pathlib import Path

root = "https://ai.stanford.edu/~amaas/data/sentiment/"
filename = "aclImdb_v1.tar.gz"
filepath = tf.keras.utils.get_file(filename, root + filename, extract=True,
                                   cache_dir=".")
if "_extracted" in filepath:
    path = Path(filepath) / "aclImdb"
else:
    path = Path(filepath).with_name("aclImdb")

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


In [4]:
path

PosixPath('datasets/aclImdb_v1_extracted/aclImdb')

In [14]:
sub_paths = path.iterdir()
for sub_path in sub_paths:
  print(sub_path)

datasets/aclImdb_v1_extracted/aclImdb/train
datasets/aclImdb_v1_extracted/aclImdb/README
datasets/aclImdb_v1_extracted/aclImdb/imdb.vocab
datasets/aclImdb_v1_extracted/aclImdb/test
datasets/aclImdb_v1_extracted/aclImdb/imdbEr.txt


In [15]:
train_dr = path / "train"
train_pos_dr = train_dr / "pos"
train_neg_dr = train_dr / "neg"
test_dr = path / "test"
test_pos_dr = test_dr / "pos"
test_neg_dr = test_dr / "neg"

In [16]:
train_pos_files = [str(path) for path in train_pos_dr.glob("*.txt")]
train_neg_files = [str(path) for path in train_neg_dr.glob("*.txt")]
test_pos_files = [str(path) for path in test_pos_dr.glob("*.txt")]
test_neg_files = [str(path) for path in test_neg_dr.glob("*.txt")]

In [17]:
print(len(train_pos_files))
print(len(train_neg_files))
print(len(test_pos_files))
print(len(test_neg_files))

12500
12500
12500
12500


Split the test set into a validation set (15,000) and a test set (10,000)

In [12]:
np.random.shuffle(test_pos_files)
np.random.shuffle(test_neg_files)

# take 5000 from each portion
valid_pos_files = test_pos_files[:5000]
test_pos_files = test_pos_files[5000:]
valid_neg_files = test_neg_files[:5000]
test_neg_files = test_neg_files[5000:]

Since the dataset fits in memory, just load all the data using tf.data.Dataset.from_tensor_slices():

In [31]:
from numpy.random import shuffle
from ast import Num
from tensorflow.data import Dataset
from typing import Sequence

def make_text_ds(dataset_files: Sequence[str]):
  # dataset_files: Sequence[str], label: Num):
  ds = Dataset.from_tensor_slices(dataset_files)
  ds = ds.map(tf.io.read_file, num_parallel_calls = tf.data.AUTOTUNE)
  return ds

def make_labeled_ds(
    pos_files: Sequence[str],
    neg_files: Sequence[str]
    shuffle=True
    cache=False):
  # dataset_files: Sequence[str], label: Num):
  pos_ds = make_text_ds(pos_files).map(lambda x: (x, 1))
  neg_ds = make_text_ds(neg_files).map(lambda x: (x, 0))
  ds = pos_ds.concatenate(neg_ds)

  if shuffle:
    ds = ds.shuffle(len(ds))

  if cache:
    ds = ds.cache()

  return ds

train_ds = make_labeled_ds(train_pos_files, train_neg_files)
valid_ds = make_labeled_ds(valid_pos_files, valid_neg_files)
test_ds = make_labeled_ds(test_pos_files, test_neg_files)

In [32]:
for x in train_ds.take(10):
  print(x[1])

tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)


## EDA